# **Практическое занятие по Pandas**

In [1]:
import pandas as pd
import numpy as np

#### **Задание 1.**

1. Создайте Series из последовательности 15 значений, равномерно разбивающих отрезок [0, 20] (воспользуйтесь функцией linspace)
2. Определите отношение элементов полученной серии к их предыдущим элементам. 
3. В результате необходимо получить среднее полученного вектора, оставив в нём только те значения, которые не более чем 1.5. 



##### Пояснения:
2. Если было бы необходимо найти последовательность из 3-х значений,  
равномерно разбивающих отрезок [0,1], то это были бы значения [0, 0.5, 1]. 

3. Если был бы дан список элементов a = [1,2,3,12], отношения элементов к предыдущим будут равны [NaN, 2, 1.5, 4]. А на последнем этапе в таком примере останется только [1.5] и среднее значение будет также 1.5.

In [2]:
# 1.1 — Series из 15 значений, равномерно разбивающих отрезок [0, 20]
s1 = pd.Series(np.linspace(0, 20, 15))
display(s1)

# 1.2 — отношение каждого элемента к предыдущему
# у первого элемента предыдущего нет, поэтому получается NaN
ratio = s1 / s1.shift(1)
display(ratio)

# 1.3 — оставляем только значения не более 1.5 и считаем среднее
filtered = ratio[ratio <= 1.5]
display(filtered)

mean_ratio = filtered.mean()
print('Ответ:', round(mean_ratio, 2))

0      0.000000
1      1.428571
2      2.857143
3      4.285714
4      5.714286
5      7.142857
6      8.571429
7     10.000000
8     11.428571
9     12.857143
10    14.285714
11    15.714286
12    17.142857
13    18.571429
14    20.000000
dtype: float64

0          NaN
1          inf
2     2.000000
3     1.500000
4     1.333333
5     1.250000
6     1.200000
7     1.166667
8     1.142857
9     1.125000
10    1.111111
11    1.100000
12    1.090909
13    1.083333
14    1.076923
dtype: float64

3     1.500000
4     1.333333
5     1.250000
6     1.200000
7     1.166667
8     1.142857
9     1.125000
10    1.111111
11    1.100000
12    1.090909
13    1.083333
14    1.076923
dtype: float64

Ответ: 1.18


#### **Задание 2.** 

По клиенту получены зашумленные данные (объект s типа Series) по его транзакциям. 

Для заданного ниже объекта s проделайте следующее:
1. Создайте новый Series, значения которого совпадают со значениями s, а индексы - целочисленные значения от 2 до 12, не включая 12. 
2. Выберите из s элементы с индексами 3 и 5, после чего просуммируйте их, сохранив результат. 
3. Выберите из s только целочисленные элементы и вычислите их дисперсию. 

Все полученные результаты округлите до 2-х знаков после запятой.

##### Пояснения:
3. Целочисленные значения - значения, имеющие тип int.
- Дисперсия рассчитывается с помощью функции из библиотеки numpy: np.var( , ddof=0) или встроенной в python функции: .var(ddof=1)

In [3]:
s = pd.Series(data=['1', 2, 3.1, 'hi!', 5, -512, 12.42, 'sber', 10.10, 98],
              index=range(6, 26, 2))

# 2.1 — новый Series с теми же значениями, но с целочисленным индексом
#       от 2 до 12, не включая 12, т.е. range(2, 12)
s2 = pd.Series(data=s.values, index=range(2, 12))
display(s2)

# 2.2 — элементы с индексами 3 и 5
selected = s2.loc[[3, 5]]
display(selected)

# сумма напрямую не считается: под индексом 5 лежит строка 'hi!'
# (TypeError: unsupported operand type(s) for +: 'int' and 'str'),
# поэтому суммируем только числовые значения
selected_sum = pd.to_numeric(selected, errors='coerce').sum()
print('Сумма:', round(selected_sum, 2))

# 2.3 — только целочисленные элементы (значения типа int) и их дисперсия
int_elements = s2[s2.apply(lambda x: isinstance(x, int))]
display(int_elements)

variance = np.var(int_elements, ddof=0)
print('Дисперсия:', round(float(variance), 2))

2         1
3         2
4       3.1
5       hi!
6         5
7      -512
8     12.42
9      sber
10     10.1
11       98
dtype: object

3      2
5    hi!
dtype: object

Сумма: 2.0


3        2
6        5
7     -512
11      98
dtype: object

Дисперсия: 57591.19


**Замечание по пункту 2.2.** В условии, по-видимому, опечатка: после выполнения п. 2.1 индекс Series равен `range(2, 12)`, и на индекс `3` приходится значение `2`, а на индекс `5` — строка `'hi!'`. Прямая сумма `[2, 'hi!']` невозможна (`TypeError`), причём это поведение одинаково и в pandas 2.2.3, и в pandas 3.x. Позиционная выборка `iloc[[3, 5]]` тоже не помогает — там `'hi!'` и `-512`.

Поэтому в ноутбуке сумма считается по числовой части отобранных значений (`pd.to_numeric(..., errors='coerce').sum()`). Если преподаватель имел в виду другие индексы, достаточно заменить `[3, 5]` — например, `[3, 6]` даст `7.0`, а `[4, 6]` — `8.1`.

#### **Задание 3.** 
1. Сгенерируйте Series из 100 значений нормально распределённой СВ (np.random.normal с дефолтными параметрами - нулевым средним и единичной дисперсией).
2. Возведите каждое значение серии в 3 степень, а значения индекса увеличьте в 3 раза. 
3. Выведите сумму элементов, строго меньших 2.6, имеющих нечётные значения индекса. 
4. Выведите количество значений серии меньше нуля.

#### Пояснения:
 - Следует внимательнее использовать [ ] для выбора данных по нескольким условиям: либо выбирать данные последовательно, либо сразу по нескольким условиям, но через оператор &. Отличие оператора and от оператора &: and - выводит последнее проверенное значение, & - выводит пересечение значений.
Пример: s[ _  &  _ ].sum() 

In [4]:
np.random.seed(242)

# 3.1 — 100 значений нормально распределённой СВ
s3 = pd.Series(np.random.normal(size=100))

# 3.2 — значения возводим в 3-ю степень, индексы увеличиваем в 3 раза
s3 = s3 ** 3
s3.index = s3.index * 3

# 3.3 — сумма значений строго меньше 2.6 с нечётными индексами
#       (&, а не and — нужно пересечение условий)
mask = (s3 < 2.6) & (s3.index % 2 != 0)
print('Сумма:', s3[mask].sum())

# 3.4 — количество значений меньше нуля
print('Количество отрицательных:', (s3 < 0).sum())

Сумма: -15.150748101821666
Количество отрицательных: 51


###***Информация для последующих заданий***
- Для всех последующих заданий будем использовать обезличенные транзакционные банковские данные. Для этого считайте в переменные tr_mcc_codes, tr_types, transactions и gender_train из одноимённых таблиц из папки data. Для таблицы transactions используйте только первые n=1000000 строк. Обратите внимание на разделители внутри каждого из файлов - они могут различаться!

In [5]:
# Для последующих заданий — обезличенные транзакционные банковские данные.
# Разделители внутри файлов различаются: в tr_types и tr_mcc_codes это ';',
# в transactions и gender_train — ','. Из transactions берём первые n = 1 000 000 строк.
N = 1_000_000

transactions = pd.read_csv('./transactions.csv', nrows=N)
tr_types = pd.read_csv('./tr_types.csv', sep=';')
tr_mcc_codes = pd.read_csv('./tr_mcc_codes.csv', sep=';')
gender_train = pd.read_csv('./gender_train.csv')

for name, df in [('transactions', transactions), ('tr_types', tr_types),
                 ('tr_mcc_codes', tr_mcc_codes), ('gender_train', gender_train)]:
    print(name, df.shape)
    display(df.head(3))

transactions (1000000, 6)


,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id
0,39026145,0 10:23:26,4814,1030,-2245.92,NaN
1,39026145,1 10:19:29,6011,7010,56147.89,NaN
2,39026145,1 10:20:56,4829,2330,-56147.89,NaN


tr_types (155, 2)


,tr_type,tr_description
0,3200,Плата за предоставление услуг посредством моби...
1,3210,Плата за предоставление отчета по счету карты ...
2,3800,Плата за обслуживание банковской карты (за пер...


tr_mcc_codes (184, 2)


,mcc_code,mcc_description
0,742,Ветеринарные услуги
1,1711,"Генеральные подрядчики по вентиляции, теплосна..."
2,1731,Подрядчики по электричеству


gender_train (8400, 2)


,customer_id,gender
0,10928546,1
1,69348468,1
2,61009479,0


## Описание данных
### Таблица ```transactions.csv```
##### Описание
Таблица содержит историю транзакций клиентов банка за один год и три месяца.

##### Формат данных

```
customer_id,tr_datetime,mcc_code,tr_type,amount,term_id
111111,15 01:40:52,1111,1000,-5224,111111
111112,15 15:18:32,3333,2000,-100,11122233
...
```
##### Описание полей

 - ```customer_id``` — идентификатор клиента;
 - ```tr_datetime``` — день и время совершения транзакции (дни нумеруются с начала данных);
 - ```mcc_code``` — mcc-код транзакции;
 - ```tr_type``` — тип транзакции;
 - ```amount``` — сумма транзакции в условных единицах со знаком; ```+``` — начисление средств клиенту (приходная транзакция), ```-``` — списание средств (расходная транзакция);
 - ```term_id``` — идентификатор терминала;


### Таблица ```gender_train.csv```

##### Описание
Данная таблица содержит информацию по полу для части клиентов, для которых он известен. Для остальных клиентов пол неизвестен.

##### Формат данных
```
customer_id,gender
111111,0
111112,1
...
```

##### Описание полей
 - ```customer_id``` — идентификатор клиента;
 - ```gender``` — пол клиента;

### Таблица ```tr_mcc_codes.csv```

##### Описание
Данная таблица содержит описание mcc-кодов транзакций.

##### Формат данных
```
mcc_code;mcc_description
1000;словесное описание mcc-кода 1000
2000;словесное описание mcc-кода 2000
...
```

##### Описание полей
 - ```mcc_code``` – mcc-код транзакции;
 - ```mcc_description``` — описание mcc-кода транзакции.

### Таблица ```tr_types.csv```

##### Описание
Данная таблица содержит описание типов транзакций.

##### Формат данных
```
tr_type;tr_description
1000;словесное описание типа транзакции 1000
2000;словесное описание типа транзакции 2000
...
```

##### Описание полей
 - ```tr_type``` – тип транзакции;
 - ```tr_description``` — описание типа транзакции;

#### **Задание 4.**
1. В tr_types выберите произвольные 100 строк с помощью метода sample (указав при этом random_seed равный 242)
2. В полученной на предыдущем этапе подвыборке найдите долю наблюдений (стобец tr_description), в которой содержится подстрока 'плата' (в любом регистре). (\*)

Выведите ответ в виде вещественного числа, округлённого до двух знаков после запятой, отделив дробную часть точкой в формате "123.45"

##### Пояснения:
(\*) Строки "ПлатА за аренду", "ПлатАза аренду", "ПЛАТА" удовлетворяют условию, так как будучи переведёнными в нижний регистр содержат подстроку "плата".

In [6]:
# 4.1 — случайная подвыборка 100 строк из tr_types
sub_types = tr_types.sample(100, random_state=242)
display(sub_types)

# 4.2 — доля строк, в описании которых встречается 'плата' в любом регистре
is_payment = sub_types['tr_description'].str.lower().str.contains('плата')

share = is_payment.mean()
print('Ответ:', f'{share:.2f}')

,tr_type,tr_description
37,6210,Возврат покупки.POS Зарубеж. ТУ
50,7034,Перевод на карту/ с карты через АТМ (без взима...
145,2431,Перевод с карты на счет физ.лица в другом банк...
4,4001,Плата за получение наличных в Сбербанке (в дру...
69,8000,Поправки по счету клиен. Зачисл.
...,...,...
24,4100,Плата за получение наличных. Россия
130,2371,Списание с карты на карту по операции <перевод...
140,2421,Перевод с карты на счет физ.лица в другом тер....
6,4011,Плата за получение наличных в АТМ Сбербанка (в...


Ответ: 0.26


#### **Задание 5.** 
1. Для поля tr_type датафрейма transactions посчитайте частоту встречаемости всех типов транзакций tr_type в transactions.
2. Из перечисленных вариантов выберите те, которые попали в топ-5 транзакций по частоте встречаемости.



In [7]:
# 5.1 — частота встречаемости всех типов транзакций
type_counts = transactions['tr_type'].value_counts()
display(type_counts)

# 5.2 — топ-5 транзакций по частоте встречаемости
display(type_counts.head(5))

tr_type
1010    231117
2010    151166
7070    149006
1110    137658
1030    118975
         ...  
2446         4
4096         4
8146         3
1510         2
8100         2
Name: count, Length: 72, dtype: int64

tr_type
1010    231117
2010    151166
7070    149006
1110    137658
1030    118975
Name: count, dtype: int64

#### **Задание 6.**
1. В датафрейме transactions задайте столбец  customer_id в качестве индекса.
2. Выделите клиента с максимальной суммой транзакции (то есть с максимальным приходом на карту). (\*)

##### Пояснения:
(\*) Если у клиента были транзакции [-10000, 10, 0, -10], то максимумом будет являться значение 10. <br>


In [8]:
# В условии путь './data/transactions.csv'; в нашей папке файлы лежат в корне.
# Читаем только первые n = 1 000 000 строк.
df = pd.read_csv('./transactions.csv', nrows=1_000_000)
display(df)

,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id
0,39026145,0 10:23:26,4814,1030,-2245.92,NaN
1,39026145,1 10:19:29,6011,7010,56147.89,NaN
2,39026145,1 10:20:56,4829,2330,-56147.89,NaN
3,39026145,1 10:39:54,5499,1010,-1392.47,NaN
4,39026145,2 15:33:42,5499,1010,-920.83,NaN
...,...,...,...,...,...,...
999995,98801178,269 10:29:06,5921,1010,-23016.59,127873
999996,98801178,270 10:26:45,5211,1010,-61380.88,10114209
999997,98801178,270 15:18:53,5921,1010,-11800.27,10105341
999998,98801178,271 12:47:13,5211,1010,-16170.59,10114209


In [9]:
# 6.1 — задаём customer_id в качестве индекса
df = df.set_index('customer_id')
display(df.head())

,tr_datetime,mcc_code,tr_type,amount,term_id
customer_id,,,,,
39026145,0 10:23:26,4814,1030,-2245.92,NaN
39026145,1 10:19:29,6011,7010,56147.89,NaN
39026145,1 10:20:56,4829,2330,-56147.89,NaN
39026145,1 10:39:54,5499,1010,-1392.47,NaN
39026145,2 15:33:42,5499,1010,-920.83,NaN


In [10]:
# 6.2 — максимальная сумма транзакции (максимальный приход на карту) по каждому клиенту
max_amount_per_client = df['amount'].groupby(level=0).max()
top_clients = max_amount_per_client.sort_values(ascending=False)
display(top_clients.head())

# клиент с максимальной из этих сумм
print('Клиент:', top_clients.index[0])
print('Сумма:', top_clients.iloc[0])

customer_id
26782922    1.459845e+08
19950741    1.347549e+08
40917212    6.737747e+07
82581973    6.737747e+07
77695202    6.123489e+07
Name: amount, dtype: float64

Клиент: 26782922
Сумма: 145984525.17


#### **Задание 7.**
Соедините transactions с всеми остальными таблицами (tr_mcc_codes, tr_types, gender_train). Причём с customers_gender_train необходимо смёрджиться с помощью left join, а с оставшимися датафреймами - через inner.
 
В результате соединения датафреймов должно получиться 999584 строки.

In [11]:
# gender_train — left join, tr_mcc_codes и tr_types — inner join
joined = (transactions
          .merge(gender_train, on='customer_id', how='left')
          .merge(tr_mcc_codes, on='mcc_code', how='inner')
          .merge(tr_types, on='tr_type', how='inner'))

print('Количество строк:', len(joined))
display(joined.columns.tolist())
display(joined.head())

Количество строк: 999584


['customer_id',
 'tr_datetime',
 'mcc_code',
 'tr_type',
 'amount',
 'term_id',
 'gender',
 'mcc_description',
 'tr_description']

,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id,gender,mcc_description,tr_description
0,39026145,0 10:23:26,4814,1030,-2245.92,NaN,1.0,"Звонки с использованием телефонов, считывающих...",Оплата услуги. Банкоматы СБ РФ
1,39026145,1 10:19:29,6011,7010,56147.89,NaN,1.0,Финансовые институты — снятие наличности автом...,Взнос наличных через АТМ (в своем тер.банке)
2,39026145,1 10:20:56,4829,2330,-56147.89,NaN,1.0,Денежные переводы,Списание с карты по операции “перевода с карты...
3,39026145,1 10:39:54,5499,1010,-1392.47,NaN,1.0,"Различные продовольственные магазины — рынки, ...",Покупка. POS ТУ СБ РФ
4,39026145,2 15:33:42,5499,1010,-920.83,NaN,1.0,"Различные продовольственные магазины — рынки, ...",Покупка. POS ТУ СБ РФ


#### **Задание 8**
Определите модуль разницы между средними тратами женщин и мужчин (трата - отрицательное значение amount). (*)

Выведите ответ в виде вещественного числа, округлённого до двух знаков после запятой, отделив дробную часть точкой в формате "123.45"

##### Пояснения:
(*) Если в результате для мужчин получились значения [-1,-3,-5], а для женщин [-1,-2,-3], то модуль разницы между средними арифметическими -3 и -2 будет равен 1.

In [12]:
# Трата — это отрицательное значение amount
spend = joined[joined['amount'] < 0]

# средние траты по полу: 0 — мужчины, 1 — женщины
mean_by_gender = spend.groupby('gender')['amount'].mean()
display(mean_by_gender)

# модуль разницы между средними тратами женщин и мужчин
diff = abs(mean_by_gender[1] - mean_by_gender[0])
print('Ответ:', f'{diff:.2f}')

gender
0.0   -53376.917122
1.0   -86094.972042
Name: amount, dtype: float64

Ответ: 32718.05


#### **Задание 9**

1. Создайте новый столбец - mcc_code+tr_type, сконкатенировав значения из соответствующих столбцов. (*)
2. Оставьте только наблюдения с отрицательным значением amount.

##### Пояснения:
(*) Для конкатенации значений в столбцах можно использовать метод .astype(str) для серии и складывать соответствующие серии. 
Либо же применять apply к строкам датафрейма, прописывая логику преобразования и конкатенации значений внутри.

In [13]:
# 9.1 — новый столбец mcc_code + tr_type через конкатенацию строк
result = transactions.assign(
    mcc_tr=transactions['mcc_code'].astype(str) + transactions['tr_type'].astype(str)
)

# 9.2 — оставляем только наблюдения с отрицательным значением amount
result = result[result['amount'] < 0]

display(result[['customer_id', 'mcc_code', 'tr_type', 'mcc_tr', 'amount']].head())
print('Строк:', len(result), '| уникальных пар mcc_code+tr_type:', result['mcc_tr'].nunique())

,customer_id,mcc_code,tr_type,mcc_tr,amount
0,39026145,4814,1030,48141030,-2245.92
2,39026145,4829,2330,48292330,-56147.89
3,39026145,5499,1010,54991010,-1392.47
4,39026145,5499,1010,54991010,-920.83
5,39026145,5541,1010,55411010,-14643.37


Строк: 769061 | уникальных пар mcc_code+tr_type: 672
